In [1]:
import sqlite3
import pandas as pd
import yfinance as yf
# Ensure bdshare is installed
try:
    from bdshare import get_basic_historical_data
except ImportError:
    import os
    os.system('pip install bdshare')
    from bdshare import get_basic_historical_data

from datetime import datetime, timedelta

# --- CONFIGURATION ---
DB_NAME = "financial_news_db.sqlite"
CSV_NAME = "market_prices_7_days.csv"

# Added UK, USA, and Europe to the configuration
MARKET_TICKERS = {
    "Italy": "FTSEMIB.MI",
    "Bangladesh": "DSEX",
    "UK": "^FTSE",          # FTSE 100
    "USA": "^GSPC",         # S&P 500
    "Europe": "^STOXX50E"   # EURO STOXX 50
}

# --- 1. DATABASE SETUP ---

def initialize_price_database(db_name):
    """Creates the SQLite database and the stock_prices table."""
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()
    cursor.execute("DROP TABLE IF EXISTS stock_prices;")
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS stock_prices (
            date TEXT NOT NULL,
            market TEXT NOT NULL,
            ticker TEXT NOT NULL,
            Open REAL,
            High REAL,
            Low REAL,
            Close REAL,
            Volume INTEGER,
            PRIMARY KEY (date, market, ticker)
        );
    """)
    conn.commit()
    conn.close()

# --- 2. DATA ACQUISITION FUNCTIONS ---

def fetch_market_data(market_name, ticker, start_date, end_date):
    """Fetches historical data for various markets including High and Low prices."""
    print(f"Fetching {market_name} ({ticker}) data...")
    try:
        if market_name == "Bangladesh":
            # Updated to use get_basic_historical_data
            data = get_basic_historical_data(ticker, start_date, end_date)
            if data is None or data.empty: return pd.DataFrame()
            data = data.rename(columns={'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close', 'volume': 'Volume'})
            data['date'] = pd.to_datetime(data['date']).dt.strftime('%Y-%m-%d')
        else:
            # yfinance for UK, USA, Europe, and Italy
            data = yf.download(ticker, start=start_date, end=end_date, interval="1d", auto_adjust=True)
            if data.empty: return pd.DataFrame()

            # Standardize multi-index columns if present
            if isinstance(data.columns, pd.MultiIndex):
                data.columns = data.columns.droplevel(1)

            data = data.reset_index()
            data['date'] = data['Date'].dt.strftime('%Y-%m-%d')

        data['market'] = market_name
        data['ticker'] = ticker
        # Explicitly include High and Low columns as requested
        return data[['date', 'market', 'ticker', 'Open', 'High', 'Low', 'Close', 'Volume']]
    except Exception as e:
        print(f"Error fetching {market_name} data: {e}")
        return pd.DataFrame()

# --- 3. COLLECTION AND EXPORT ---

def collect_and_export_prices(db_name, csv_name):
    initialize_price_database(db_name)

    # Calculate exactly 7 days of historical data
    end_dt = datetime.now()
    start_dt = end_dt - timedelta(days=7)
    start_date = start_dt.strftime('%Y-%m-%d')
    end_date = end_dt.strftime('%Y-%m-%d')

    all_data = []
    for market, ticker in MARKET_TICKERS.items():
        df = fetch_market_data(market, ticker, start_date, end_date)
        if not df.empty:
            all_data.append(df)

    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)

        # Save to SQLite
        conn = sqlite3.connect(db_name)
        combined_df.to_sql('stock_prices', conn, if_exists='append', index=False)

        # Save to CSV
        combined_df.to_csv(csv_name, index=False)
        print(f"\nSuccessfully saved 7 days of data to '{csv_name}'.")

        # Display sample output
        print("\n--- Final Results Preview ---")
        print(combined_df.sort_values(by='date', ascending=False).head(10))
        conn.close()
    else:
        print("No data was collected.")

if __name__ == "__main__":
    collect_and_export_prices(DB_NAME, CSV_NAME)

Fetching Italy (FTSEMIB.MI) data...


[*********************100%***********************]  1 of 1 completed


Fetching Bangladesh (DSEX) data...
Error fetching Bangladesh data: No basic historical data found.
Fetching UK (^FTSE) data...


[*********************100%***********************]  1 of 1 completed


Fetching USA (^GSPC) data...


[*********************100%***********************]  1 of 1 completed


Fetching Europe (^STOXX50E) data...


[*********************100%***********************]  1 of 1 completed


Successfully saved 7 days of data to 'market_prices_7_days.csv'.

--- Final Results Preview ---
Price        date  market      ticker          Open          High  \
19     2026-03-04  Europe   ^STOXX50E   5780.250000   5888.209961   
4      2026-03-04   Italy  FTSEMIB.MI  44446.000000  45517.000000   
9      2026-03-04      UK       ^FTSE  10483.900391  10588.799805   
14     2026-03-04     USA       ^GSPC   6831.689941   6885.939941   
3      2026-03-03   Italy  FTSEMIB.MI  45593.000000  45648.000000   
18     2026-03-03  Europe   ^STOXX50E   5937.330078   5937.330078   
8      2026-03-03      UK       ^FTSE  10780.299805  10780.299805   
13     2026-03-03     USA       ^GSPC   6800.259766   6840.049805   
2      2026-03-02   Italy  FTSEMIB.MI  46325.000000  46519.000000   
7      2026-03-02      UK       ^FTSE  10911.000000  10911.000000   

Price           Low         Close      Volume  
19      5772.250000   5870.919922    32429500  
4      44328.000000  45337.000000   573520400  

In [2]:
from google.colab import files

files.download('market_prices_7_days.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>